# Policy Gradients: From REINFORCE to Actor-Critic


Value-based methods — as developed in [01-rl-foundations.html](./01-rl-foundations.html) — learn an action-value function $Q^*(s, a)$ and extract a policy by acting greedily. Policy gradient methods flip this: they parameterize $\pi_\theta(a \mid s)$ directly and optimize by gradient ascent on the expected return $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G_0].$ The policy is the primary object; no value function is strictly required.

There are compelling reasons for this. When the action space is large or continuous — think robotic joint torques, or the vocabulary of an LLM — maintaining a table or network over all $(s, a)$ pairs becomes unwieldy. A parameterized policy maps directly from states to action probabilities (or distributions), sidestepping this explosion. Stochastic policies also provide natural exploration: the agent does not need an $\varepsilon$-greedy heuristic; it simply samples from $\pi_\theta.$ And crucially, the objective $J(\theta)$ is differentiable with respect to $\theta$, so we can use gradient ascent directly on what we care about.

This notebook derives the **policy gradient theorem**, implements **REINFORCE** from scratch on CartPole-v1, then progressively reduces variance via a **learned baseline** and finally **actor-critic** (TD advantages). Each step introduces exactly one idea to stabilize training. By the end, we have a working A2C implementation trained entirely on short rollouts, without waiting for episode completion.

**Prerequisites:** [01-rl-foundations.html](./01-rl-foundations.html). We use PyTorch for neural networks and `gymnasium` for CartPole-v1.


## Why Policy Gradients?


Q-learning finds the optimal $Q^*$ and then reads off the policy as $\pi(s) = \operatorname{argmax}_a Q^*(s, a).$ This works beautifully when the action space is a small discrete set, but breaks down in two regimes: (1) when actions are continuous (so $\operatorname{argmax}$ requires a separate optimization at each step), and (2) when the optimal policy is stochastic — for instance in partially observed environments where randomness is genuinely useful. Policy gradient methods address both issues at once by treating the policy itself as the learnable object.

The comparison below summarizes the key differences:

| | Value-based | Policy-based |
|---|---|---|
| What is learned | $Q^*(s, a)$, policy is derived | $\pi_\theta(a \mid s)$ directly |
| Action spaces | Discrete, small | Discrete or continuous |
| Stochasticity | Deterministic policy | Natural |
| Example algorithms | Q-learning, DQN | REINFORCE, PPO |

LLM alignment is an extreme instance of the policy gradient setting: the action space is the full vocabulary (~100K tokens), a trajectory is the complete generated response, and the "environment" is human preferences encoded in a reward model. Algorithms like GRPO and PPO are direct applications of the framework we develop here — the only difference in scale, not in principle.


## The Policy Gradient Theorem


We want to maximize the expected return over trajectories $\tau = (s_0, a_0, r_0, s_1, a_1, r_1, \ldots)$ sampled from $\pi_\theta$:

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)] = \int p_\theta(\tau)\, R(\tau)\, d\tau.$$

Taking the gradient with respect to $\theta$:

$$\nabla_\theta J(\theta) = \int \nabla_\theta p_\theta(\tau)\, R(\tau)\, d\tau.$$

This is intractable to evaluate directly — we cannot differentiate through an integral over all trajectories. The **log-derivative trick** converts it into an expectation:

$$\nabla_\theta p_\theta(\tau) = p_\theta(\tau)\, \nabla_\theta \log p_\theta(\tau),$$

so $\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[\nabla_\theta \log p_\theta(\tau) \cdot R(\tau)].$ Now we need $\nabla_\theta \log p_\theta(\tau).$ Expanding the trajectory probability:

$$\log p_\theta(\tau) = \log p(s_0) + \sum_{t=0}^{T-1} \left[\log \pi_\theta(a_t \mid s_t) + \log P(s_{t+1} \mid s_t, a_t)\right].$$

The initial state distribution $\log p(s_0)$ and the environment dynamics $\log P(s_{t+1} \mid s_t, a_t)$ do not depend on $\theta$, so their gradients vanish. Only the policy terms survive:

$$\nabla_\theta \log p_\theta(\tau) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t).$$

Substituting back gives the **policy gradient theorem**:

$$\boxed{\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\!\left[\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot R(\tau)\right].}$$

We can sharpen this further. Past rewards at time $t' < t$ cannot be influenced by the action $a_t$ taken at time $t$, so they contribute zero in expectation to the gradient. Replacing $R(\tau)$ with the **reward-to-go** $G_t = \sum_{t'=t}^{T} r_{t'}$:

$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}\!\left[\sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot G_t\right].$$

This is still unbiased but has lower variance than using the full trajectory return.

:::{.callout-important}
The log-derivative trick $\nabla_\theta \log \pi_\theta(a \mid s) = \nabla_\theta \pi_\theta(a \mid s) / \pi_\theta(a \mid s)$ converts an intractable gradient of an expectation into an expectation of gradients — estimable by sampling. This same identity underlies every LLM alignment gradient (DPO, GRPO, PPO).

:::


## Setup and CartPole


We work with **CartPole-v1**: a 4-dimensional continuous state space (cart position, cart velocity, pole angle, pole angular velocity) and 2 discrete actions (push left or push right). An episode terminates when the pole tips past $\pm 12°$, the cart exits $\pm 2.4\text{m}$, or 500 steps are reached. The environment is considered **solved** when a rolling average reward of $\geq 475$ over 100 consecutive episodes is achieved. The simplicity of CartPole lets us focus entirely on the algorithmic differences between REINFORCE, REINFORCE with baseline, and actor-critic, without GPU requirements.


**Setup.** We import libraries and fix random seeds for reproducibility:


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline
backend_inline.set_matplotlib_formats("svg")

torch.manual_seed(42)
np.random.seed(42)


**Environment check.** We instantiate CartPole-v1 and inspect its spaces:


In [ ]:
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=42)
print(f"Observation space: {env.observation_space}")
print(f"Action space:      {env.action_space}")
print(f"Initial obs:       {obs}")
env.close()


**Rollout utility.** A helper that collects one full episode given any policy function:


In [ ]:
def rollout(env, policy_fn, seed=None):
    """Collect one episode. Returns list of (state, action, reward) triples."""
    obs, _ = env.reset(seed=seed)
    trajectory = []
    done = False
    while not done:
        action = policy_fn(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        trajectory.append((obs, action, reward))
        obs = next_obs
        done = terminated or truncated
    return trajectory


## REINFORCE


REINFORCE is the simplest policy gradient algorithm. We parameterize $\pi_\theta(a \mid s)$ as a neural network, sample complete trajectories, compute the gradient estimate, and update $\theta$ by gradient ascent. Using the reward-to-go form, the Monte Carlo estimator averaged over $N$ episodes is:

$$\hat{g} = \frac{1}{N} \sum_{n=1}^{N} \sum_{t=0}^{T_n - 1} \nabla_\theta \log \pi_\theta(a_t^n \mid s_t^n) \cdot G_t^n.$$

In practice we maximize $J(\theta)$ by minimizing the negative: $\mathcal{L} = -\frac{1}{N}\sum_{n,t} \log \pi_\theta(a_t^n \mid s_t^n) \cdot G_t^n.$ PyTorch's autograd then computes $\hat{g}$ automatically.


**Policy network.** A two-hidden-layer MLP that outputs a softmax probability distribution over actions:


In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, obs_dim=4, hidden_dim=64, n_actions=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

    def act(self, obs):
        """Sample action from policy distribution; return action and log prob."""
        obs_t = torch.FloatTensor(obs).unsqueeze(0)
        probs = self(obs_t)
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)


**Returns.** We compute discounted reward-to-go $G_t = \sum_{t'=t}^{T} \gamma^{t'-t} r_{t'}$ by iterating in reverse:


In [ ]:
def compute_returns(rewards, gamma=0.99):
    """Compute discounted reward-to-go for each timestep."""
    G = 0
    returns = []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return torch.FloatTensor(returns)


**REINFORCE training loop.** We collect one episode per update step, compute returns, normalize them, and take a gradient step:


In [ ]:
def train_reinforce(n_episodes=1000, gamma=0.99, lr=1e-2, log_interval=100):
    env = gym.make("CartPole-v1")
    policy = PolicyNet()
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)

    episode_returns = []

    for ep in range(n_episodes):
        # Collect one episode
        obs, _ = env.reset()
        log_probs, rewards = [], []
        done = False
        while not done:
            action, log_prob = policy.act(obs)
            obs, r, terminated, truncated, _ = env.step(action)
            log_probs.append(log_prob)
            rewards.append(r)
            done = terminated or truncated

        episode_returns.append(sum(rewards))

        # Compute returns and policy gradient loss
        returns = compute_returns(rewards, gamma)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)  # <1>

        log_probs = torch.stack(log_probs)
        loss = -(log_probs * returns).mean()                           # <2>

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (ep + 1) % log_interval == 0:
            avg = np.mean(episode_returns[-100:])
            print(f"Episode {ep+1:4d}  avg_return={avg:.1f}")

    env.close()
    return policy, episode_returns


1. Normalizing returns within each episode reduces gradient variance significantly. This is a heuristic baseline — §5 introduces the principled version via a learned value function.
2. The REINFORCE loss is $\mathcal{L} = -\sum_t \log \pi_\theta(a_t \mid s_t)\, G_t$; maximizing $J(\theta)$ corresponds to minimizing this negative quantity.


**Training.** We run REINFORCE for 1000 episodes and record the results:


In [ ]:
policy_reinforce, returns_reinforce = train_reinforce(n_episodes=1000)


**Figure.** Learning curve for plain REINFORCE. The 100-episode rolling average shows the overall trend, while the raw episode returns illustrate the high variance:


In [ ]:
#| code-fold: true
def smooth(x, window=100):
    """Rolling mean with a given window size."""
    return np.convolve(x, np.ones(window) / window, mode="valid")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(returns_reinforce, alpha=0.25, color="steelblue", linewidth=0.8, label="Episode return")
ax.plot(
    np.arange(99, len(returns_reinforce)),
    smooth(returns_reinforce),
    color="steelblue", linewidth=2, label="100-ep rolling avg"
)
ax.axhline(475, color="tomato", linestyle="--", linewidth=1.2, label="Solved (475)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("REINFORCE on CartPole-v1")
ax.legend(loc="upper left")
ax.grid(linestyle="dotted", alpha=0.5)
plt.tight_layout()
plt.show()


REINFORCE learns, but is noisy and slow. The variance comes from using full Monte Carlo returns $G_t$: each $G_t$ averages over all future randomness in the trajectory, including luck of the environment that has nothing to do with the quality of action $a_t.$ We address this in §5.


## Baselines and Variance Reduction


Any function $b(s)$ that depends only on the state (not the action) is called a **baseline**. Subtracting it from the return does not bias the gradient:

$$\mathbb{E}_{a \sim \pi_\theta}\!\left[\nabla_\theta \log \pi_\theta(a \mid s) \cdot b(s)\right] = b(s) \cdot \nabla_\theta \underbrace{\sum_a \pi_\theta(a \mid s)}_{=1} = 0.$$

So the gradient estimate $\hat{g} = \sum_t \nabla_\theta \log \pi_\theta(a_t \mid s_t) \cdot (G_t - b(s_t))$ remains unbiased for any choice of $b.$ The **advantage function** is:

$$A^\pi(s_t, a_t) = G_t - V^\pi(s_t),$$

where $V^\pi(s_t) = \mathbb{E}_{\pi}[G_t \mid s_t]$ is the expected return from state $s_t$ under $\pi.$ The optimal baseline that minimizes variance is $b^*(s) = V^\pi(s).$ We approximate this with a learned value network $V_\phi$ trained alongside the policy.

:::{.callout-note}
In GRPO, the baseline is the group mean $\bar{r} = \frac{1}{G}\sum_{i=1}^G r_i$. This is a Monte Carlo estimate of $V^\pi(x) = \mathbb{E}_{y \sim \pi}[r(x, y)]$ — the expected reward over all possible responses to prompt $x$. The same principle applies: subtract an estimate of state value to obtain an advantage.

:::


**Value network.** A separate MLP $V_\phi\colon \mathbb{R}^4 \to \mathbb{R}$ that estimates the state value:


In [ ]:
class ValueNet(nn.Module):
    def __init__(self, obs_dim=4, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


**REINFORCE with baseline.** We add $V_\phi$ and compute advantages $\hat{A}_t = G_t - V_\phi(s_t)$ at each timestep. The value network is trained by minimizing the mean-squared error against the observed returns:


In [ ]:
def train_reinforce_baseline(n_episodes=1000, gamma=0.99, lr=1e-2, log_interval=100):
    env = gym.make("CartPole-v1")
    policy = PolicyNet()
    value_net = ValueNet()
    policy_opt = torch.optim.Adam(policy.parameters(), lr=lr)
    value_opt = torch.optim.Adam(value_net.parameters(), lr=lr)

    episode_returns = []

    for ep in range(n_episodes):
        obs, _ = env.reset()
        log_probs, rewards, states = [], [], []
        done = False
        while not done:
            states.append(obs)
            action, log_prob = policy.act(obs)
            obs, r, terminated, truncated, _ = env.step(action)
            log_probs.append(log_prob)
            rewards.append(r)
            done = terminated or truncated

        episode_returns.append(sum(rewards))

        returns = compute_returns(rewards, gamma)               # <1>
        states_t = torch.FloatTensor(np.array(states))
        values = value_net(states_t)                            # <2>
        advantages = (returns - values.detach())                # <3>
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        log_probs_t = torch.stack(log_probs)
        policy_loss = -(log_probs_t * advantages).mean()
        value_loss = F.mse_loss(values, returns)                # <4>

        policy_opt.zero_grad()
        policy_loss.backward()
        policy_opt.step()

        value_opt.zero_grad()
        value_loss.backward()
        value_opt.step()

        if (ep + 1) % log_interval == 0:
            avg = np.mean(episode_returns[-100:])
            print(f"Episode {ep+1:4d}  avg_return={avg:.1f}")

    env.close()
    return policy, episode_returns


1. We still compute full Monte Carlo returns $G_t$ — REINFORCE with baseline remains an on-policy Monte Carlo method.
2. $V_\phi(s_t)$ approximates the expected return from state $s_t,$ serving as the baseline.
3. We detach the value estimates when computing advantages so that the policy gradient does not flow into the value network through this path.
4. The value network is trained separately to minimize $\mathcal{L}_V = \frac{1}{T}\sum_t (G_t - V_\phi(s_t))^2.$


**Training.** We run REINFORCE with baseline and compare against plain REINFORCE:


In [ ]:
policy_baseline, returns_baseline = train_reinforce_baseline(n_episodes=1000)


**Figure.** Comparing REINFORCE and REINFORCE with learned value baseline. Both 100-episode rolling averages are shown:


In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(7, 3.5))

r1 = np.array(returns_reinforce)
r2 = np.array(returns_baseline)
n = min(len(r1), len(r2))

ax.plot(r1[:n], alpha=0.15, color="steelblue", linewidth=0.6)
ax.plot(r2[:n], alpha=0.15, color="darkorange", linewidth=0.6)
ax.plot(
    np.arange(99, n),
    smooth(r1[:n]),
    color="steelblue", linewidth=2, label="REINFORCE"
)
ax.plot(
    np.arange(99, n),
    smooth(r2[:n]),
    color="darkorange", linewidth=2, label="REINFORCE + baseline"
)
ax.axhline(475, color="tomato", linestyle="--", linewidth=1.2, label="Solved (475)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("Variance Reduction via Baseline")
ax.legend(loc="upper left")
ax.grid(linestyle="dotted", alpha=0.5)
plt.tight_layout()
plt.show()


## Actor-Critic


REINFORCE — even with a baseline — uses full Monte Carlo returns $G_t$: we must wait until the episode ends before computing any gradient. **Actor-Critic** methods replace $G_t$ with a **bootstrapped TD target**, enabling online updates after every step (or short rollout). The tradeoff is a small amount of bias from the imperfect critic, but variance drops substantially.

The **one-step TD advantage** (TD error $\delta_t$) is:

$$\hat{A}_t = r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t) = \delta_t.$$

This is the core of the **Advantage Actor-Critic (A2C)** algorithm. The **actor** (policy $\pi_\theta$) is updated to maximize the expected advantage; the **critic** (value function $V_\phi$) is updated to minimize the Bellman residual. The combined objective on a batch of $T$ steps is:

$$\mathcal{L} = \underbrace{-\frac{1}{T}\sum_{t=0}^{T-1} \log \pi_\theta(a_t \mid s_t) \cdot \hat{A}_t}_{\text{policy loss}} + \underbrace{\frac{c_1}{T}\sum_{t=0}^{T-1}\left(V_\phi(s_t) - G_t^{\text{ret}}\right)^2}_{\text{value loss}} - \underbrace{c_2 \cdot H[\pi_\theta(\cdot \mid s_t)]}_{\text{entropy bonus}},$$

with $c_1 = 0.5$ and $c_2 = 0.01.$ The **entropy bonus** $H[\pi] = -\sum_a \pi \log \pi$ encourages the policy to remain exploratory and prevents premature convergence to a deterministic solution.


**A2C training loop.** We collect short rollouts of fixed length (5 steps), compute $n$-step returns, and update after each rollout:


In [ ]:
def train_a2c(n_updates=5000, rollout_len=5, gamma=0.99, lr=3e-4,
              c1=0.5, c2=0.01, log_interval=500):
    env = gym.make("CartPole-v1")
    policy = PolicyNet()
    value_net = ValueNet()
    optimizer = torch.optim.Adam(
        list(policy.parameters()) + list(value_net.parameters()), lr=lr
    )

    obs, _ = env.reset(seed=42)
    current_return = 0
    all_returns = []

    for update in range(n_updates):
        # Collect rollout_len steps
        states, rewards, dones, log_probs_list = [], [], [], []

        for _ in range(rollout_len):
            action, log_prob = policy.act(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            states.append(obs)
            rewards.append(reward)
            dones.append(done)
            log_probs_list.append(log_prob)

            current_return += reward
            obs = next_obs
            if done:
                all_returns.append(current_return)
                current_return = 0
                obs, _ = env.reset()

        # Compute n-step returns and advantages
        states_t = torch.FloatTensor(np.array(states))
        next_obs_t = torch.FloatTensor(next_obs).unsqueeze(0)

        with torch.no_grad():
            next_value = value_net(next_obs_t) * (1 - float(done))  # <1>

        returns = []
        R = next_value.item()
        for r, d in zip(reversed(rewards), reversed(dones)):
            R = r + gamma * R * (1 - float(d))
            returns.insert(0, R)
        returns_t = torch.FloatTensor(returns)

        values = value_net(states_t)
        advantages = returns_t - values.detach()                     # <2>

        log_probs_t = torch.stack(log_probs_list)

        # Entropy bonus
        probs = policy(states_t)
        entropy = -(probs * probs.log()).sum(dim=-1).mean()           # <3>

        policy_loss = -(log_probs_t * advantages).mean()
        value_loss = F.mse_loss(values, returns_t)
        loss = policy_loss + c1 * value_loss - c2 * entropy

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(                              # <4>
            list(policy.parameters()) + list(value_net.parameters()), 0.5
        )
        optimizer.step()

        if (update + 1) % log_interval == 0 and len(all_returns) >= 100:
            avg = np.mean(all_returns[-100:])
            print(f"Update {update+1:5d}  avg_return={avg:.1f}  episodes={len(all_returns)}")

    env.close()
    return policy, all_returns


1. Bootstrap from the next state value; zero it out if the last step was a terminal state, since there is no future reward after episode termination.
2. We detach the critic's value estimates when computing advantages to prevent gradients flowing into the critic through this path — the critic is trained separately via `value_loss`.
3. Entropy of the policy distribution: $H[\pi] = -\sum_a \pi(a \mid s) \log \pi(a \mid s).$ The negative sign in the loss turns maximizing entropy into a regularizer that discourages collapse to a deterministic policy.
4. Gradient clipping prevents exploding gradients, which are common in on-policy RL with small batches and bootstrapped targets.


**Training.** We run A2C for 5000 update steps, each consuming 5 environment steps:


In [ ]:
policy_a2c, returns_a2c = train_a2c(n_updates=5000, rollout_len=5)


**Figure.** Comparing all three algorithms by 100-episode rolling average return:


In [ ]:
#| code-fold: true
fig, ax = plt.subplots(figsize=(7, 3.5))

colors = {"REINFORCE": "steelblue", "REINFORCE + baseline": "darkorange", "A2C": "seagreen"}
all_data = [
    (returns_reinforce, "REINFORCE"),
    (returns_baseline, "REINFORCE + baseline"),
    (returns_a2c, "A2C"),
]

for data, label in all_data:
    arr = np.array(data)
    ax.plot(arr, alpha=0.12, color=colors[label], linewidth=0.6)
    if len(arr) >= 100:
        ax.plot(
            np.arange(99, len(arr)),
            smooth(arr),
            color=colors[label], linewidth=2, label=label
        )

ax.axhline(475, color="tomato", linestyle="--", linewidth=1.2, label="Solved (475)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return")
ax.set_title("REINFORCE vs. REINFORCE + Baseline vs. A2C")
ax.legend(loc="upper left", fontsize=8)
ax.grid(linestyle="dotted", alpha=0.5)
plt.tight_layout()
plt.show()


A2C uses short rollouts of 5 steps instead of full episodes, enabling more frequent updates. The TD advantages have lower variance than Monte Carlo returns but introduce bias from the imperfect critic. It turns out that controlling this bias-variance tradeoff precisely is the key to further improvement. The next notebook introduces **Generalized Advantage Estimation** (GAE), which blends one-step TD and Monte Carlo returns via an exponential weighting parameter $\lambda \in [0, 1]$, and **Proximal Policy Optimization** (PPO) to prevent destabilizing policy updates.

:::{.callout-caution}
A2C with very short rollouts can underperform REINFORCE on simple environments like CartPole if the critic is insufficiently trained. The bootstrapped targets create a moving learning signal for the critic, and with only 5-step rollouts the $n$-step returns may carry significant bias early in training. In practice, A2C shines on harder tasks where episode lengths are too long for Monte Carlo methods to be practical.

:::


## Appendix: The Variance Reduction Landscape


The methods developed in this notebook occupy specific positions in the bias-variance tradeoff of advantage estimation. The table below situates each algorithm and previews what the next two notebooks introduce:

| Method | Advantage Estimate | Variance | Bias | Sample Efficiency |
|---|---|---|---|---|
| REINFORCE | $G_t$ (full return) | High | None | Low |
| REINFORCE + baseline | $G_t - V_\phi(s_t)$ | Medium | None | Low |
| A2C (TD) | $r_t + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$ | Low | Yes (critic) | High |
| GAE (next notebook) | Exponentially-weighted blend | Tunable | Tunable | High |
| PPO (next notebook) | GAE with clipped updates | Low | Tunable | High |

The fundamental tension is that lower variance requires more reliance on the critic (bootstrapping), which introduces bias proportional to the critic's approximation error. GAE controls this with a single parameter $\lambda$: at $\lambda = 0$ we recover one-step TD (low variance, high bias); at $\lambda = 1$ we recover Monte Carlo (no bias, high variance). PPO then adds a mechanism to prevent the policy from updating too aggressively in a single step, which is the remaining instability in A2C.


---


■
